In [1]:
!pip install googlemaps flask


  Preparing metadata (setup.py) ... done
  Created wheel for googlemaps: filename=googlemaps-4.10.0-py3-none-any.whl size=40714 sha256=53b20230cf151d91263a72540d4d9d9fc5e33a3f2e2b7d219ecdb380327b7fca
  Stored in directory: /root/.cache/pip/wheels/4c/6a/a7/bbc6f5c200032025ee655deb5e163ce8594fa05e67d973aad6
Successfully built googlemaps


In [3]:
# Direct assignment (less secure for sharing notebooks)
API_KEY = ''

# Or set as environment variable in Colab
import os
os.environ['GOOGLE_API_KEY'] = API_KEY


In [5]:
import os

API_KEY = os.getenv('GOOGLE_API_KEY')
if not API_KEY:
    raise ValueError("Google API Key not found. Check your Kaggle Secrets setup.")

import googlemaps
gmaps = googlemaps.Client(key=API_KEY)


In [6]:
import googlemaps

API_KEY = ''
gmaps = googlemaps.Client(key=API_KEY)

def find_nearby_places(location, keyword, radius=5000):
    # location: (lat, lng)
    places_result = gmaps.places_nearby(location=location, keyword=keyword, radius=radius)
    return places_result.get('results', [])

def get_route(origin, destination):
    directions = gmaps.directions(origin, destination, mode='driving')
    return directions[0] if directions else None


In [7]:
import sqlite3

conn = sqlite3.connect('agent_memory.db')
c = conn.cursor()
c.execute('''CREATE TABLE IF NOT EXISTS sessions
             (user_id TEXT, session_id TEXT, request TEXT, response TEXT)''')
conn.commit()

def save_session(user_id, session_id, request, response):
    c.execute('INSERT INTO sessions VALUES (?, ?, ?, ?)', (user_id, session_id, request, response))
    conn.commit()

def get_user_sessions(user_id):
    c.execute('SELECT * FROM sessions WHERE user_id=?', (user_id,))
    return c.fetchall()


In [8]:
from multiprocessing import Process, Queue

def agent_communicator(q_in, q_out):
    while True:
        msg = q_in.get()
        if msg == 'STOP':
            break
        # Process message and respond or forward
        response = f"Agent received: {msg}"
        q_out.put(response)

q_main_to_agent = Queue()
q_agent_to_main = Queue()
p = Process(target=agent_communicator, args=(q_main_to_agent, q_agent_to_main))
p.start()

# Send a message to agent
q_main_to_agent.put('Need volunteer driver for hospital pickup.')
print(q_agent_to_main.get())

# Stop process when done
q_main_to_agent.put('STOP')
p.join()


Agent received: Need volunteer driver for hospital pickup.


In [9]:
def handle_user_request(user_id, session_id, location, request_text):
    # Simple urgency detection
    urgent_keywords = ['emergency', 'urgent', 'help', 'asap']
    is_urgent = any(word in request_text.lower() for word in urgent_keywords)

    # Fetch relevant nearby resources, e.g. hospitals
    if is_urgent:
        places = find_nearby_places(location, 'hospital')
        top_place = places[0] if places else None
        response = f"Found nearest hospital: {top_place['name']}" if top_place else "No nearby hospital found."
    else:
        response = "How can I assist you today?"

    # Save session record
    save_session(user_id, session_id, request_text, response)

    # Simulate agent-agent communication for urgent requests
    if is_urgent:
        q_main_to_agent.put(f"Urgent help requested by {user_id}")

    return response


In [12]:
def find_nearby_places(location, keyword, radius=5000):
    # Mocked response
    return [
        {'name': 'Mock Hospital 1', 'location': (40.714, -74.005)},
        {'name': 'Mock Hospital 2', 'location': (40.715, -74.006)}
    ]


In [13]:
# Sample static dataset
hospitals = [
    {'name': 'City Hospital', 'lat': 40.713, 'lng': -74.005},
    {'name': 'Downtown Clinic', 'lat': 40.712, 'lng': -74.006}
]

def find_nearby_static(location, keyword):
    # Dummy function: return static data
    return hospitals


In [14]:
import sqlite3

# Connect to (or create) dummy in-memory database for demonstration
conn = sqlite3.connect(':memory:')
c = conn.cursor()

# Create table for resources (e.g., hospitals)
c.execute('''
CREATE TABLE resources (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    type TEXT NOT NULL,
    latitude REAL,
    longitude REAL
)
''')

# Insert dummy resource data
dummy_resources = [
    (1, 'Mock City Hospital', 'hospital', 40.713, -74.005),
    (2, 'Downtown Clinic', 'hospital', 40.712, -74.006),
    (3, 'Neighborhood Shelter', 'shelter', 40.715, -74.007)
]
c.executemany('INSERT INTO resources VALUES (?, ?, ?, ?, ?)', dummy_resources)
conn.commit()

# Create table for sessions
c.execute('''
CREATE TABLE sessions (
    user_id TEXT NOT NULL,
    session_id TEXT NOT NULL,
    request TEXT,
    response TEXT
)
''')
conn.commit()


In [15]:
#2. Define functions to fetch dummy nearby resources and save sessions

def find_nearby_resources(resource_type, radius=5000):
    # Simulate fetching nearby resources filtered by type
    # We don't actually use radius here for simplicity
    c.execute('SELECT name, latitude, longitude FROM resources WHERE type=?', (resource_type,))
    return c.fetchall()

def save_session(user_id, session_id, request, response):
    c.execute('INSERT INTO sessions (user_id, session_id, request, response) VALUES (?, ?, ?, ?)',
              (user_id, session_id, request, response))
    conn.commit()

def get_user_sessions(user_id):
    c.execute('SELECT * FROM sessions WHERE user_id=?', (user_id,))
    return c.fetchall()


In [16]:
#3. Use these functions in your original handle_user_request logic with dummy output
def handle_user_request(user_id, session_id, location, request_text):
    urgent_keywords = ['emergency', 'urgent', 'help', 'asap', 'hospital']
    is_urgent = any(word in request_text.lower() for word in urgent_keywords)

    if is_urgent:
        # Use dummy resource fetch instead of Google API
        resources = find_nearby_resources('hospital')
        if resources:
            top_resource = resources[0]
            response = f"Found nearest hospital: {top_resource[0]} at location {top_resource[1]}, {top_resource[2]}"
        else:
            response = "No nearby hospital found."
    else:
        response = "How can I assist you today?"

    save_session(user_id, session_id, request_text, response)

    return response


In [17]:
#4. Test with sample user request and print session data
user_loc = (40.7128, -74.0060)  # Example lat/lng not used in dummy code here
print(handle_user_request('user123', 'sess001', user_loc, 'I need urgent help at hospital!'))

# Retrieve all sessions for the user to demonstrate persistent memory
sessions = get_user_sessions('user123')
for s in sessions:
    print(s)


Found nearest hospital: Mock City Hospital at location 40.713, -74.005
('user123', 'sess001', 'I need urgent help at hospital!', 'Found nearest hospital: Mock City Hospital at location 40.713, -74.005')
